三個模型定義（LSTM/GRU用RMSE點預測，Transformer是自己刻的Encoder-only架構，同樣RMSE點預測，跟原論文3.2節架構一致但加了building embedding以符合跨建築比較的公平性）

Cell 1 - 讀資料

In [ ]:
import pandas as pd

train = pd.read_parquet('dataset/train.parquet')
val = pd.read_parquet('dataset/val.parquet')
test = pd.read_parquet('dataset/test.parquet')
test_unseen = pd.read_parquet('dataset/test_unseen.parquet')

seen_full = pd.concat([train, val, test], ignore_index=True)
seen_full = seen_full.sort_values(['building_id', 'timestamp']).reset_index(drop=True)
seen_full['time_idx'] = seen_full.groupby('building_id').cumcount()

test_unseen_full = test_unseen.sort_values(['building_id', 'timestamp']).reset_index(drop=True)
test_unseen_full['time_idx'] = test_unseen_full.groupby('building_id').cumcount()

print("seen_full筆數：", len(seen_full))

In [ ]:
import numpy as np

n_bad = (seen_full['load_intensity'] == 0.0).sum()
print(f"修正前，load_intensity剛好是0.0的筆數: {n_bad} ({n_bad/len(seen_full)*100:.3f}%)")

# 把字面上0.0的值視為缺值，改成NaN，再依建築分組做線性插值
seen_full.loc[seen_full['load_intensity'] == 0.0, 'load_intensity'] = np.nan
seen_full['load_intensity'] = (
    seen_full.groupby('building_id')['load_intensity']
    .transform(lambda s: s.interpolate(method='linear').ffill().bfill())
)

n_remaining = seen_full['load_intensity'].isna().sum()
print(f"修正後，仍為NaN的筆數: {n_remaining}（應該是0）")
print(f"修正後，load_intensity的0.0筆數: {(seen_full['load_intensity']==0.0).sum()}")

Cell 2 - 建立training_dataset（跟TFT完全一樣的設定）

In [ ]:
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer

max_encoder_length = 168
max_prediction_length = 24

for col in ['building_id', 'site_id', 'hour', 'weekday', 'is_weekend', 'month']:
    seen_full[col] = seen_full[col].astype(str)

training_cutoff = seen_full[seen_full['timestamp'] < '2017-06-01']['time_idx'].max()

training_dataset = TimeSeriesDataSet(
    seen_full[lambda x: x.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="load_intensity",
    group_ids=["building_id"],
    min_encoder_length=max_encoder_length // 2,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,
    static_categoricals=["building_id", "site_id"],
    static_reals=["sqm"],
    time_varying_known_categoricals=["hour", "weekday", "is_weekend", "month"],
    time_varying_known_reals=["time_idx", "airTemperature", "dewTemperature", "windSpeed", "windDirection"],
    time_varying_unknown_reals=["load_intensity"],
    target_normalizer=GroupNormalizer(groups=["building_id"], transformation="softplus"),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)
print("訓練資料集樣本數：", len(training_dataset))

Cell 3 - validation_dataset + dataloader

In [ ]:
validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset, seen_full, predict=True, stop_randomization=True
)

train_dataloader = training_dataset.to_dataloader(
    train=True, batch_size=512, num_workers=4,
    persistent_workers=True, pin_memory=True,
)
val_dataloader = validation_dataset.to_dataloader(
    train=False, batch_size=1024, num_workers=2,
    persistent_workers=True, pin_memory=True,
)
print("train batch數：", len(train_dataloader), "val batch數：", len(val_dataloader))

In [ ]:
from pytorch_forecasting import RecurrentNetwork
from pytorch_forecasting.metrics import RMSE

lstm_model = RecurrentNetwork.from_dataset(
    training_dataset,
    cell_type="LSTM",
    hidden_size=32,
    rnn_layers=2,
    dropout=0.1,
    loss=RMSE(),
    learning_rate=0.03,
)

print(f"LSTM模型參數量: {sum(p.numel() for p in lstm_model.parameters()):,}")

In [ ]:
import lightning.pytorch as pl

smoke_trainer = pl.Trainer(
    max_epochs=1,
    accelerator="gpu",
    devices=1,
    limit_train_batches=5,
    limit_val_batches=1,
    enable_model_summary=False,
    logger=False,
    enable_checkpointing=False,
)

smoke_trainer.fit(
    lstm_model,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)

print("LSTM smoke test 完成，沒有報錯即代表架構跑得通")

In [ ]:
gru_model = RecurrentNetwork.from_dataset(
    training_dataset,
    cell_type="GRU",
    hidden_size=32,
    rnn_layers=2,
    dropout=0.1,
    loss=RMSE(),
    learning_rate=0.03,
)

print(f"GRU模型參數量: {sum(p.numel() for p in gru_model.parameters()):,}")

GRU baseline

In [ ]:
gru_model = RecurrentNetwork.from_dataset(
    training_dataset,
    cell_type="GRU",
    hidden_size=32,
    rnn_layers=2,
    dropout=0.1,
    loss=RMSE(),
    learning_rate=0.03,
)

print(f"GRU模型參數量: {sum(p.numel() for p in gru_model.parameters()):,}")

In [ ]:
smoke_trainer_gru = pl.Trainer(
    max_epochs=1,
    accelerator="gpu",
    devices=1,
    limit_train_batches=5,
    limit_val_batches=1,
    enable_model_summary=False,
    logger=False,
    enable_checkpointing=False,
)

smoke_trainer_gru.fit(
    gru_model,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)

print("GRU smoke test 完成")

In [ ]:
batch = next(iter(train_dataloader))
x, y = batch

print("x的key：", list(x.keys()))
for k, v in x.items():
    if hasattr(v, 'shape'):
        print(f"  {k}: {v.shape}")
    else:
        print(f"  {k}: {v}")

print("\ny[0]（target）形狀：", y[0].shape)

In [ ]:
print([attr for attr in dir(training_dataset) if 'categ' in attr.lower()])

In [ ]:
cat_names = ["building_id", "site_id", "hour", "weekday", "is_weekend", "month"]
cat_sizes = [len(training_dataset._categorical_encoders[name].classes_) for name in cat_names]

for name, size in zip(cat_names, cat_sizes):
    print(f"{name}: {size} 個類別")

In [ ]:
import math
import torch
import torch.nn as nn

class VanillaTransformerNet(nn.Module):
    def __init__(self, cat_sizes, n_continuous, d_model=64, nhead=4,
                 num_layers=2, dim_feedforward=128, dropout=0.1, prediction_length=24):
        super().__init__()
        emb_dims = [min(50, (s + 1) // 2) for s in cat_sizes]
        self.embeddings = nn.ModuleList([nn.Embedding(s, d) for s, d in zip(cat_sizes, emb_dims)])

        total_input_dim = sum(emb_dims) + n_continuous
        self.input_projection = nn.Linear(total_input_dim, d_model)

        pe = torch.zeros(500, d_model)
        position = torch.arange(500).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pos_encoding", pe)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_projection = nn.Linear(d_model, prediction_length)

    def forward(self, encoder_cat, encoder_cont):
        embedded = [emb(encoder_cat[..., i]) for i, emb in enumerate(self.embeddings)]
        embedded = torch.cat(embedded, dim=-1)
        combined = torch.cat([embedded, encoder_cont], dim=-1)
        projected = self.input_projection(combined)
        seq_len = projected.size(1)
        projected = projected + self.pos_encoding[:seq_len].unsqueeze(0)
        encoded = self.encoder(projected)
        return self.output_projection(encoded[:, -1, :])

net = VanillaTransformerNet(cat_sizes=cat_sizes, n_continuous=11)
out = net(x["encoder_cat"], x["encoder_cont"])
print("輸出形狀：", out.shape, "（應該要是 [512, 24]）")

LightningModule

In [ ]:
import lightning.pytorch as pl

class VanillaTransformerModel(pl.LightningModule):
    def __init__(self, cat_sizes, n_continuous, d_model=64, nhead=4,
                 num_layers=2, dim_feedforward=128, dropout=0.1,
                 prediction_length=24, learning_rate=0.001):
        super().__init__()
        self.save_hyperparameters()
        self.net = VanillaTransformerNet(
            cat_sizes=cat_sizes, n_continuous=n_continuous, d_model=d_model,
            nhead=nhead, num_layers=num_layers, dim_feedforward=dim_feedforward,
            dropout=dropout, prediction_length=prediction_length,
        )
        self.learning_rate = learning_rate

    def forward(self, x):
        return self.net(x["encoder_cat"], x["encoder_cont"])

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = torch.sqrt(nn.functional.mse_loss(y_hat, y[0]) + 1e-8)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = torch.sqrt(nn.functional.mse_loss(y_hat, y[0]) + 1e-8)
        self.log("val_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

transformer_model = VanillaTransformerModel(cat_sizes=cat_sizes, n_continuous=11)
print(f"Transformer模型參數量: {sum(p.numel() for p in transformer_model.parameters()):,}")

In [ ]:
smoke_trainer_transformer = pl.Trainer(
    max_epochs=1,
    accelerator="gpu",
    devices=1,
    limit_train_batches=5,
    limit_val_batches=1,
    enable_model_summary=False,
    logger=False,
    enable_checkpointing=False,
)

smoke_trainer_transformer.fit(
    transformer_model,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)

print("Transformer smoke test 完成")

In [ ]:
# 用數值型態判斷，避免字串比較出錯
hour_int = seen_full['hour'].astype(int)
is_weekend_int = seen_full['is_weekend'].astype(int)

seen_full['is_offpeak'] = (is_weekend_int == 1) | (hour_int < 9) | (hour_int >= 17)

print("off-peak時段筆數比例：", seen_full['is_offpeak'].mean())

In [ ]:
train_mask = seen_full['timestamp'] < '2017-06-01'

def compute_offpeak_baseline(group):
    offpeak_vals = group.loc[group['is_offpeak'], 'load_intensity']
    q1, q3 = offpeak_vals.quantile([0.25, 0.75])
    iqr = q3 - q1
    return offpeak_vals.median() + 1.5 * iqr

offpeak_baseline = (
    seen_full[train_mask]
    .groupby('building_id')
    .apply(compute_offpeak_baseline)
)

print("前5棟建築的off-peak基線：")
print(offpeak_baseline.head())
print("\n基線統計描述：")
print(offpeak_baseline.describe())

In [ ]:
seen_full['offpeak_baseline'] = seen_full['building_id'].map(offpeak_baseline)
seen_full['anomaly_schedule'] = seen_full['is_offpeak'] & (seen_full['load_intensity'] > seen_full['offpeak_baseline'])

print("時段違常標記筆數：", seen_full['anomaly_schedule'].sum())
print("佔全體資料比例：", seen_full['anomaly_schedule'].mean())
print("佔off-peak資料比例：", seen_full.loc[seen_full['is_offpeak'], 'anomaly_schedule'].mean())

In [ ]:
eui = (
    seen_full[train_mask]
    .groupby('building_id')
    .agg(EUI=('load_intensity', 'sum'), sqm=('sqm', 'first'), site_id=('site_id', 'first'))
    .reset_index()
)

print("EUI統計：")
print(eui['EUI'].describe())
print("\n每棟建築sqm範圍：", eui['sqm'].min(), "~", eui['sqm'].max())

In [ ]:
eui['sqm_quartile'] = pd.qcut(eui['sqm'], q=4, labels=['Q1_small', 'Q2', 'Q3', 'Q4_large'])

print("各規模組建築數：")
print(eui['sqm_quartile'].value_counts().sort_index())

print("\n各規模組sqm範圍：")
print(eui.groupby('sqm_quartile')['sqm'].agg(['min', 'max']))

In [ ]:
eui['peer_group'] = eui['site_id'].astype(str) + '_' + eui['sqm_quartile'].astype(str)
eui['peer_percentile'] = eui.groupby('peer_group')['EUI'].rank(pct=True)
eui['anomaly_peer'] = eui['peer_percentile'] >= 0.75

print("同儕落後標記建築數：", eui['anomaly_peer'].sum(), "／", len(eui))

print("\n各peer_group的建築數分布（檢查有沒有池子太小）：")
peer_group_sizes = eui.groupby('peer_group').size()
print(peer_group_sizes.describe())
print("\n建築數<=2的peer_group數量：", (peer_group_sizes <= 2).sum())

In [ ]:
import numpy as np
# 標記哪些建築的peer_group太小，需要退回較寬鬆的分組
eui['peer_group_size'] = eui['peer_group'].map(peer_group_sizes)
MIN_GROUP_SIZE = 4

small_group_mask = eui['peer_group_size'] < MIN_GROUP_SIZE
print(f"peer_group建築數<{MIN_GROUP_SIZE}的建築數：", small_group_mask.sum())

# 對這些建築，改用只依site_id分組的百分位（放寬規模限制）
eui['fallback_percentile'] = eui.groupby('site_id')['EUI'].rank(pct=True)

# 合併：優先用細分組百分位，小池子改用site_id百分位
eui['final_percentile'] = np.where(small_group_mask, eui['fallback_percentile'], eui['peer_percentile'])
eui['anomaly_peer'] = eui['final_percentile'] >= 0.75

print("\n修正後同儕落後標記建築數：", eui['anomaly_peer'].sum(), "／", len(eui))

In [ ]:
flagged_examples = eui[eui['anomaly_peer']].sort_values('final_percentile', ascending=False).head(5)
print("同儕落後建築範例（EUI最高的5棟）：")
print(flagged_examples[['building_id', 'site_id', 'sqm_quartile', 'EUI', 'peer_group_size', 'final_percentile']])

# 對照同組其他建築的EUI，確認真的比較高
example_building = flagged_examples.iloc[0]
peer_group_id = example_building['peer_group'] if example_building['peer_group_size'] >= MIN_GROUP_SIZE else None
if peer_group_id:
    print(f"\n{example_building['building_id']}所在peer_group的其他建築EUI比較：")
    print(eui[eui['peer_group'] == peer_group_id][['building_id', 'EUI']].sort_values('EUI', ascending=False))

In [ ]:
target = eui[eui['building_id'] == 'Bull_office_Ella'].iloc[0]
print(f"Bull_office_Ella 所在peer_group: {target['peer_group']}，組內建築數: {target['peer_group_size']}")
print("\n同組所有建築EUI比較：")
print(eui[eui['peer_group'] == target['peer_group']][['building_id', 'sqm', 'EUI']].sort_values('EUI', ascending=False))

# 順便看一下這棟建築的load_intensity時間趨勢分布，確認是持續性高耗能還是少數極端值拉高總和
building_data = seen_full[(seen_full['building_id'] == 'Bull_office_Ella') & train_mask]
print(f"\nBull_office_Ella load_intensity描述統計：")
print(building_data['load_intensity'].describe())

In [ ]:
schedule_examples = seen_full[seen_full['anomaly_schedule']].sample(5, random_state=42)
print(schedule_examples[['building_id', 'timestamp', 'hour', 'is_weekend', 'load_intensity', 'offpeak_baseline']])

In [ ]:
# 用數值型態判斷，避免字串比較出錯
hour_int = seen_full['hour'].astype(int)
is_weekend_int = seen_full['is_weekend'].astype(int)

seen_full['is_offpeak'] = (is_weekend_int == 1) | (hour_int < 9) | (hour_int >= 17)

print("off-peak時段筆數比例：", seen_full['is_offpeak'].mean())

### 注入測試 (驗證異常偵測方法有效性的標準做法)

引用了ASHRAE RP-1043等建築故障診斷文獻的「分級嚴重度測試」做法佐證這個方法論站得住腳。

規則二尖峰型注入測試

In [ ]:
import numpy as np
np.random.seed(42)

test_mask = seen_full['timestamp'] >= '2017-10-01'

candidates = seen_full[test_mask & seen_full['is_offpeak'] & 
                        (seen_full['load_intensity'] < seen_full['offpeak_baseline'])].copy()
print("候選乾淨off-peak資料點數:", len(candidates))

N_PER_LEVEL = 60
severities = {'輕度(1.7x)': 1.7, '中度(2.3x)': 2.3, '重度(5x)': 5.0}
results_rule2_spike = []

for label, mult in severities.items():
    spike_idx = np.random.choice(candidates.index, size=N_PER_LEVEL, replace=False)
    control_idx = np.random.choice(candidates.index.difference(spike_idx), size=N_PER_LEVEL, replace=False)

    injected_vals = seen_full.loc[spike_idx, 'load_intensity'] * mult
    tp = (injected_vals.values > seen_full.loc[spike_idx, 'offpeak_baseline'].values).sum()
    fp = (seen_full.loc[control_idx, 'load_intensity'].values > seen_full.loc[control_idx, 'offpeak_baseline'].values).sum()

    recall = tp / N_PER_LEVEL
    precision = tp / (tp + fp) if (tp+fp) > 0 else np.nan
    results_rule2_spike.append({'類型':'尖峰型','強度':label,'recall':recall,'precision':precision,'TP':int(tp),'FP':int(fp)})

import pandas as pd
print(pd.DataFrame(results_rule2_spike))

規則二持續型（24-48小時區間）注入測試

In [ ]:
test_df = seen_full[test_mask].copy().sort_values(['building_id','timestamp']).reset_index(drop=True)
buildings = test_df['building_id'].dropna().unique().tolist()

N_EPISODES = 40
DURATION_H = 36
rng = np.random.default_rng(42)
results_rule2_sustained = []

for label, mult in severities.items():
    tp_total, n_offpeak_total = 0, 0
    control_flagged, control_total = 0, 0
    chosen, attempts = 0, 0

    while chosen < N_EPISODES and attempts < 2000:
        attempts += 1
        b = rng.choice(buildings)
        b_data = test_df[test_df['building_id'] == b]
        if len(b_data) < DURATION_H + 5:
            continue
        start = rng.integers(0, len(b_data) - DURATION_H)
        window = b_data.iloc[start:start+DURATION_H]
        offpeak_window = window[window['is_offpeak']]
        if len(offpeak_window) == 0:
            continue

        injected_vals = offpeak_window['load_intensity'] * mult
        tp_total += (injected_vals.values > offpeak_window['offpeak_baseline'].values).sum()
        n_offpeak_total += len(offpeak_window)

        c_start = rng.integers(0, len(b_data) - DURATION_H)
        c_window = b_data.iloc[c_start:c_start+DURATION_H]
        c_offpeak = c_window[c_window['is_offpeak']]
        if len(c_offpeak) > 0:
            control_flagged += (c_offpeak['load_intensity'].values > c_offpeak['offpeak_baseline'].values).sum()
            control_total += len(c_offpeak)
        chosen += 1

    recall = tp_total / n_offpeak_total if n_offpeak_total else np.nan
    precision = tp_total / (tp_total + control_flagged) if (tp_total + control_flagged) > 0 else np.nan
    results_rule2_sustained.append({'類型':'持續型(36h)','強度':label,'注入案例數':chosen,
                                      'recall':round(recall,3),'precision':round(precision,3)})

print(pd.DataFrame(results_rule2_sustained))

規則三的建築層級測試

In [ ]:
MIN_GROUP_SIZE = 4
severities3 = {'輕度(1.7x)': 1.7, '中度(2.3x)': 2.3, '重度(5x)': 5.0}

candidates3 = eui[eui['final_percentile'] < 0.5]['building_id'].tolist()
print("候選正常建築數:", len(candidates3))

N_PER_LEVEL3 = 20
results_rule3 = []

for label, mult in severities3.items():
    chosen = rng.choice(candidates3, size=N_PER_LEVEL3, replace=False)
    tp = 0

    for b in chosen:
        b_peer_group = eui.loc[eui['building_id']==b, 'peer_group'].values[0]
        b_group_size = eui.loc[eui['building_id']==b, 'peer_group_size'].values[0]
        b_site = eui.loc[eui['building_id']==b, 'site_id'].values[0]
        use_fallback = b_group_size < MIN_GROUP_SIZE

        eui_copy = eui.copy()
        orig_eui = eui_copy.loc[eui_copy['building_id']==b, 'EUI'].values[0]
        eui_copy.loc[eui_copy['building_id']==b, 'EUI'] = orig_eui * mult

        if use_fallback:
            group_members = eui_copy[eui_copy['site_id']==b_site].copy()
        else:
            group_members = eui_copy[eui_copy['peer_group']==b_peer_group].copy()

        group_members['new_percentile'] = group_members['EUI'].rank(pct=True)
        b_new_pct = group_members.loc[group_members['building_id']==b, 'new_percentile'].values[0]
        tp += int(b_new_pct >= 0.75)

    recall = tp / N_PER_LEVEL3
    results_rule3.append({'強度':label, '注入建築數':N_PER_LEVEL3, 'recall':round(recall,3)})

print(pd.DataFrame(results_rule3))

載入comparison.parquet並跟規則二做交叉比對

In [ ]:
import pandas as pd

comp = pd.read_parquet('comparison.parquet')  # 如果路徑不同請調整
comp['timestamp'] = pd.to_datetime(comp['timestamp'])

merged = seen_full.merge(comp, on=['building_id','timestamp'], how='inner')
print("合併後筆數:", len(merged), " (原seen_full筆數:", len(seen_full), ")")

ct = pd.crosstab(merged['anomaly_schedule'], merged['official_anomaly_flag'])
print("\n交叉表（列=規則二判定, 欄=官方判定）:")
print(ct)

tp = ((merged['anomaly_schedule']) & (merged['official_anomaly_flag'])).sum()
rule2_total = merged['anomaly_schedule'].sum()
official_total = merged['official_anomaly_flag'].sum()
print(f"\n規則二標記數: {rule2_total}, 官方標記數: {official_total}, 重疊數: {tp}")

接下來驗證缺失值bug（確認2.9%的load_intensity=0跟官方缺失標記的關係

In [ ]:
print("load_intensity恰好為0的筆數:", (merged['load_intensity']==0).sum(),
      f"（佔比 {(merged['load_intensity']==0).mean():.4%}）")

ct2 = pd.crosstab(merged['load_intensity']==0, merged['official_anomaly_flag'])
print("\n交叉表（列=load_intensity是否為0, 欄=official_anomaly_flag）:")
print(ct2)

subset = merged[(merged['load_intensity']==0) & (merged['official_anomaly_flag'])]
print("\n交集中 meter_reading_cleaned 為 NaN 的比例:", subset['meter_reading_cleaned'].isna().mean())

規則三的t檢定驗證

In [ ]:
!pip install scipy

from scipy import stats

official_rate = merged.groupby('building_id', observed=True)['official_anomaly_flag'].mean()
eui['official_missing_rate'] = eui['building_id'].map(official_rate)

g1 = eui[eui['anomaly_peer']]['official_missing_rate'].dropna()
g2 = eui[~eui['anomaly_peer']]['official_missing_rate'].dropna()
t, p = stats.ttest_ind(g1, g2, equal_var=False)
print(f"t檢定: t={t:.3f}, p={p:.4f}")

corr = eui[['EUI','official_missing_rate']].corr().iloc[0,1]
print(f"EUI 與 官方缺失率 的相關係數: {corr:.4f}")

個案質性檢視

In [ ]:
import numpy as np
np.random.seed(7)

real_flagged = seen_full[seen_full['anomaly_schedule']]
sample2 = real_flagged.sample(8, random_state=7)
sample2 = sample2.assign(超出基線倍數=(sample2['load_intensity']/sample2['offpeak_baseline']))

print("=== 規則二：真實案例抽樣（8筆）===")
print(sample2[['building_id','timestamp','weekday','hour','is_weekend','load_intensity','offpeak_baseline','超出基線倍數']].to_string(index=False))

print()
np.random.seed(11)
flagged_buildings = eui[eui['anomaly_peer']]
sample3 = flagged_buildings.sample(7, random_state=11)

print("=== 規則三：真實案例抽樣（7棟）===")
print(sample3[['building_id','site_id','sqm','sqm_quartile','EUI','peer_group_size','final_percentile']].to_string(index=False))

## LSTM（第一個baseline）

In [ ]:
batch = next(iter(train_dataloader))
x, y = batch

overall_min = x["encoder_cont"].min().item()
overall_max = x["encoder_cont"].max().item()
print(f"這個batch的encoder_cont整體極值: min={overall_min:.3f}, max={overall_max:.3f}")

print("\n各變數明細:")
for i, name in enumerate(training_dataset.reals):
    col = x["encoder_cont"][..., i]
    print(f"{name}: min={col.min().item():.3f}, max={col.max().item():.3f}")

# 額外掃描更多batch，找出真正出現極端值的那個batch
print("\n掃描前200個batch，找出encoder_cont min < -20 的batch:")
extreme_batches = []
for i, (bx, by) in enumerate(train_dataloader):
    bmin = bx["encoder_cont"].min().item()
    if bmin < -20:
        extreme_batches.append((i, bmin))
    if i >= 200:
        break
print("找到的極端batch:", extreme_batches[:10])

In [ ]:
print("continuous變數順序:", training_dataset.reals)

# 找出encoder_cont中，哪一個特徵欄位的極值最誇張
ec = x["encoder_cont"]  # 沿用上一格已經取出的batch
for i, name in enumerate(training_dataset.reals):
    col = ec[..., i]
    print(f"{name}: min={col.min().item():.3f}, max={col.max().item():.3f}")

In [ ]:
from lightning.pytorch.tuner import Tuner

tuning_trainer = pl.Trainer(accelerator="gpu", devices=1, gradient_clip_val=0.1)
tuner = Tuner(tuning_trainer)

lr_finder = tuner.lr_find(
    lstm_model,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
    min_lr=1e-5,
    max_lr=1e-1,
)
new_lr = lr_finder.suggestion()
print("建議的learning rate：", new_lr)
lstm_model.hparams.learning_rate = new_lr

In [ ]:
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger

early_stop_callback = EarlyStopping(
    monitor="val_loss", min_delta=1e-4, patience=3, verbose=True, mode="min"
)
checkpoint_callback = ModelCheckpoint(
    dirpath="lstm_checkpoints",
    filename="best-lstm-{epoch:02d}-val_loss={val_loss:.4f}",
    monitor="val_loss", mode="min", save_top_k=1,
)
lr_logger = LearningRateMonitor()
logger = TensorBoardLogger("lightning_logs", name="lstm")

trainer = pl.Trainer(
    max_epochs=30,
    accelerator="gpu",
    devices=1,
    gradient_clip_val=0.1,
    callbacks=[early_stop_callback, checkpoint_callback, lr_logger],
    logger=logger,
    enable_progress_bar=True,
    log_every_n_steps=10,
)

print("開始正式訓練 LSTM ...")
trainer.fit(
    lstm_model,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)
print("訓練完成")
print("最佳模型儲存位置：", checkpoint_callback.best_model_path)
print("最佳val_loss：", checkpoint_callback.best_model_score.item())

## GRU

In [ ]:
from pytorch_forecasting import RecurrentNetwork
from pytorch_forecasting.metrics import RMSE

gru_model = RecurrentNetwork.from_dataset(
    training_dataset,
    cell_type="GRU",
    hidden_size=32,
    rnn_layers=2,
    dropout=0.1,
    loss=RMSE(),
    learning_rate=0.03,
)

from lightning.pytorch.tuner import Tuner

tuning_trainer = pl.Trainer(accelerator="gpu", devices=1, gradient_clip_val=0.1)
tuner = Tuner(tuning_trainer)

lr_finder = tuner.lr_find(
    gru_model,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
    min_lr=1e-5,
    max_lr=1e-1,
)
new_lr = lr_finder.suggestion()
print("建議的learning rate：", new_lr)
gru_model.hparams.learning_rate = new_lr

In [ ]:
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger

early_stop_callback = EarlyStopping(
    monitor="val_loss", min_delta=1e-4, patience=3, verbose=True, mode="min"
)
checkpoint_callback = ModelCheckpoint(
    dirpath="gru_checkpoints",
    filename="best-gru-{epoch:02d}-{val_loss:.4f}",
    monitor="val_loss", mode="min", save_top_k=1,
)
lr_logger = LearningRateMonitor()
logger = TensorBoardLogger("lightning_logs", name="gru")

trainer = pl.Trainer(
    max_epochs=30,
    accelerator="gpu",
    devices=1,
    gradient_clip_val=0.1,
    callbacks=[early_stop_callback, checkpoint_callback, lr_logger],
    logger=logger,
    enable_progress_bar=True,
    log_every_n_steps=10,
)

print("開始正式訓練 GRU ...")
trainer.fit(
    gru_model,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)
print("訓練完成")
print("最佳模型儲存位置：", checkpoint_callback.best_model_path)
print("最佳val_loss：", checkpoint_callback.best_model_score.item())

In [ ]:
from lightning.pytorch.tuner import Tuner

tuning_trainer = pl.Trainer(accelerator="gpu", devices=1, gradient_clip_val=0.1)
tuner = Tuner(tuning_trainer)

lr_finder = tuner.lr_find(
    transformer_model,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
    min_lr=1e-5,
    max_lr=1e-1,
)
new_lr = lr_finder.suggestion()
print("建議的learning rate：", new_lr)
transformer_model.learning_rate = new_lr

In [ ]:
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger

early_stop_callback = EarlyStopping(
    monitor="val_loss", min_delta=1e-4, patience=3, verbose=True, mode="min"
)
checkpoint_callback = ModelCheckpoint(
    dirpath="transformer_checkpoints",
    filename="best-transformer-{epoch:02d}-{val_loss:.4f}",
    monitor="val_loss", mode="min", save_top_k=1,
)
lr_logger = LearningRateMonitor()
logger = TensorBoardLogger("lightning_logs", name="transformer")

trainer = pl.Trainer(
    max_epochs=30,
    accelerator="gpu",
    devices=1,
    gradient_clip_val=0.1,
    callbacks=[early_stop_callback, checkpoint_callback, lr_logger],
    logger=logger,
    enable_progress_bar=True,
    log_every_n_steps=10,
)

print("開始正式訓練 Transformer ...")
trainer.fit(
    transformer_model,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)
print("訓練完成")
print("最佳模型儲存位置：", checkpoint_callback.best_model_path)
print("最佳val_loss：", checkpoint_callback.best_model_score.item())

#### 建立 training_dataset + 載入 LSTM checkpoint + 評估 test 集

In [ ]:
import pandas as pd
import torch
from pytorch_forecasting import TimeSeriesDataSet, RecurrentNetwork
from pytorch_forecasting.data import GroupNormalizer

# 1. 重新載入資料，組回seen_full
train = pd.read_parquet("dataset/train.parquet")
val = pd.read_parquet("dataset/val.parquet")
test = pd.read_parquet("dataset/test.parquet")
seen_full = pd.concat([train, val, test], ignore_index=True)
seen_full['timestamp'] = pd.to_datetime(seen_full['timestamp'])
seen_full = seen_full.sort_values(['building_id', 'timestamp']).reset_index(drop=True)
seen_full['time_idx'] = (
    (seen_full['timestamp'] - seen_full['timestamp'].min())
    .dt.total_seconds().div(3600).astype(int)
)

# 2. 重新套用缺失值修正（跟訓練LSTM/GRU/Transformer時一樣的那段）
import numpy as np
seen_full.loc[seen_full['load_intensity'] == 0.0, 'load_intensity'] = np.nan
seen_full['load_intensity'] = (
    seen_full.groupby('building_id')['load_intensity']
    .transform(lambda s: s.interpolate(method='linear').ffill().bfill())
)
print("修正後仍為NaN的筆數:", seen_full['load_intensity'].isna().sum())

# 3. 類別型欄位轉字串
for col in ["building_id", "site_id", "hour", "weekday", "is_weekend", "month"]:
    seen_full[col] = seen_full[col].astype(str)

# 4. 建立training_dataset
max_encoder_length = 168
max_prediction_length = 24
training_cutoff = seen_full[seen_full['timestamp'] < '2017-06-01']['time_idx'].max()

training_dataset = TimeSeriesDataSet(
    seen_full[lambda x: x.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="load_intensity",
    group_ids=["building_id"],
    min_encoder_length=max_encoder_length // 2,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,
    static_categoricals=["building_id", "site_id"],
    static_reals=["sqm"],
    time_varying_known_categoricals=["hour", "weekday", "is_weekend", "month"],
    time_varying_known_reals=["time_idx", "airTemperature", "dewTemperature", "windSpeed", "windDirection"],
    time_varying_unknown_reals=["load_intensity"],
    target_normalizer=GroupNormalizer(groups=["building_id"], transformation="softplus"),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

# 5. 從checkpoint載入LSTM模型
lstm_model = RecurrentNetwork.load_from_checkpoint(
    r"C:\Users\peggy\Desktop\能源\lstm_checkpoints\best-lstm-epoch=05-val_loss=val_loss=0.0171.ckpt"
)
lstm_model.eval()
if torch.cuda.is_available():
    lstm_model = lstm_model.cuda()

# 6. 建立test_dataset並評估
test_start_time_idx = seen_full[seen_full['timestamp'] >= '2017-10-01']['time_idx'].min()

test_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    seen_full,
    predict=False,
    stop_randomization=True,
    min_prediction_idx=test_start_time_idx,
    min_prediction_length=max_prediction_length,
)
print("test_dataset樣本數：", len(test_dataset))

test_dataloader = test_dataset.to_dataloader(train=False, batch_size=128, num_workers=0)

predictions = lstm_model.predict(test_dataloader, mode="prediction", return_y=True)
y_pred = predictions.output.cpu()
y_true = predictions.y[0].cpu()

print("y_pred有NaN:", torch.isnan(y_pred).any().item())
rmse = torch.sqrt(torch.mean((y_pred - y_true) ** 2)).item()
mae = torch.mean(torch.abs(y_pred - y_true)).item()
print(f"LSTM — test set RMSE: {rmse:.4f}, MAE: {mae:.4f}")

In [ ]:
from copy import deepcopy

# 1. 重新載入test_unseen_full
test_unseen_full = pd.read_parquet("dataset/test_unseen.parquet")
test_unseen_full["timestamp"] = pd.to_datetime(test_unseen_full["timestamp"])
test_unseen_full = test_unseen_full.sort_values(["building_id", "timestamp"]).reset_index(drop=True)
test_unseen_full["time_idx"] = (
    (test_unseen_full["timestamp"] - test_unseen_full["timestamp"].min())
    .dt.total_seconds().div(3600).astype(int)
)
for col in ["building_id", "site_id", "hour", "weekday", "is_weekend", "month"]:
    test_unseen_full[col] = test_unseen_full[col].astype(str)

# 2. 對LSTM做平均embedding補丁
building_embed_layer = lstm_model.embeddings.embeddings["building_id"]
mean_embedding = building_embed_layer.weight.data.mean(dim=0, keepdim=True)
old_weight = building_embed_layer.weight.data
new_weight = torch.cat([old_weight, mean_embedding], dim=0)

new_embed_layer = torch.nn.Embedding(new_weight.shape[0], new_weight.shape[1])
new_embed_layer.weight.data = new_weight
if torch.cuda.is_available():
    new_embed_layer = new_embed_layer.cuda()
lstm_model.embeddings.embeddings["building_id"] = new_embed_layer

patched_encoder = deepcopy(training_dataset._categorical_encoders["building_id"])
unseen_building_ids = test_unseen_full["building_id"].unique()
n_new = 0
for bid in unseen_building_ids:
    if bid not in patched_encoder.classes_:
        patched_encoder.classes_[bid] = new_weight.shape[0] - 1
        n_new += 1
print(f"新增了{n_new}個building_id對應到index {new_weight.shape[0]-1}（應該是28）")

# 3. 建立test_unseen_dataset並評估
test_unseen_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    test_unseen_full,
    predict=False,
    stop_randomization=True,
    categorical_encoders={"building_id": patched_encoder},
    min_prediction_length=max_prediction_length,
)
print("test_unseen_dataset樣本數：", len(test_unseen_dataset))

test_unseen_dataloader = test_unseen_dataset.to_dataloader(train=False, batch_size=128, num_workers=0)

predictions_unseen = lstm_model.predict(test_unseen_dataloader, mode="prediction", return_y=True)
y_pred_unseen = predictions_unseen.output.cpu()
y_true_unseen = predictions_unseen.y[0].cpu()

print("y_pred_unseen有NaN:", torch.isnan(y_pred_unseen).any().item())
rmse_unseen = torch.sqrt(torch.mean((y_pred_unseen - y_true_unseen) ** 2)).item()
mae_unseen = torch.mean(torch.abs(y_pred_unseen - y_true_unseen)).item()
print(f"LSTM — test_unseen set RMSE: {rmse_unseen:.4f}, MAE: {mae_unseen:.4f}")

### GRU的 test 評估

In [ ]:
from pytorch_forecasting import RecurrentNetwork

gru_model = RecurrentNetwork.load_from_checkpoint(
    r"C:\Users\peggy\Desktop\能源\gru_checkpoints\best-gru-epoch=15-val_loss=0.0147.ckpt"
)
gru_model.eval()
if torch.cuda.is_available():
    gru_model = gru_model.cuda()

predictions = gru_model.predict(test_dataloader, mode="prediction", return_y=True)
y_pred = predictions.output.cpu()
y_true = predictions.y[0].cpu()

print("y_pred有NaN:", torch.isnan(y_pred).any().item())
rmse = torch.sqrt(torch.mean((y_pred - y_true) ** 2)).item()
mae = torch.mean(torch.abs(y_pred - y_true)).item()
print(f"GRU — test set RMSE: {rmse:.4f}, MAE: {mae:.4f}")

### GRU 的 test_unseen

In [ ]:
building_embed_layer = gru_model.embeddings.embeddings["building_id"]
mean_embedding = building_embed_layer.weight.data.mean(dim=0, keepdim=True)
old_weight = building_embed_layer.weight.data
new_weight = torch.cat([old_weight, mean_embedding], dim=0)

new_embed_layer = torch.nn.Embedding(new_weight.shape[0], new_weight.shape[1])
new_embed_layer.weight.data = new_weight
if torch.cuda.is_available():
    new_embed_layer = new_embed_layer.cuda()
gru_model.embeddings.embeddings["building_id"] = new_embed_layer

patched_encoder_gru = deepcopy(training_dataset._categorical_encoders["building_id"])
unseen_building_ids = test_unseen_full["building_id"].unique()
n_new = 0
for bid in unseen_building_ids:
    if bid not in patched_encoder_gru.classes_:
        patched_encoder_gru.classes_[bid] = new_weight.shape[0] - 1
        n_new += 1
print(f"新增了{n_new}個building_id對應到index {new_weight.shape[0]-1}（應該是28）")

test_unseen_dataset_gru = TimeSeriesDataSet.from_dataset(
    training_dataset,
    test_unseen_full,
    predict=False,
    stop_randomization=True,
    categorical_encoders={"building_id": patched_encoder_gru},
    min_prediction_length=max_prediction_length,
)
print("test_unseen_dataset樣本數：", len(test_unseen_dataset_gru))

test_unseen_dataloader_gru = test_unseen_dataset_gru.to_dataloader(train=False, batch_size=128, num_workers=0)

predictions_unseen = gru_model.predict(test_unseen_dataloader_gru, mode="prediction", return_y=True)
y_pred_unseen = predictions_unseen.output.cpu()
y_true_unseen = predictions_unseen.y[0].cpu()

print("y_pred_unseen有NaN:", torch.isnan(y_pred_unseen).any().item())
rmse_unseen = torch.sqrt(torch.mean((y_pred_unseen - y_true_unseen) ** 2)).item()
mae_unseen = torch.mean(torch.abs(y_pred_unseen - y_true_unseen)).item()
print(f"GRU — test_unseen set RMSE: {rmse_unseen:.4f}, MAE: {mae_unseen:.4f}")

In [ ]:
import math
import torch
import torch.nn as nn
import lightning.pytorch as pl

class VanillaTransformerNet(nn.Module):
    def __init__(self, cat_sizes, n_continuous, d_model=64, nhead=4,
                 num_layers=2, dim_feedforward=128, dropout=0.1, prediction_length=24):
        super().__init__()
        emb_dims = [min(50, (s + 1) // 2) for s in cat_sizes]
        self.embeddings = nn.ModuleList([nn.Embedding(s, d) for s, d in zip(cat_sizes, emb_dims)])

        total_input_dim = sum(emb_dims) + n_continuous
        self.input_projection = nn.Linear(total_input_dim, d_model)

        pe = torch.zeros(500, d_model)
        position = torch.arange(500).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pos_encoding", pe)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_projection = nn.Linear(d_model, prediction_length)

    def forward(self, encoder_cat, encoder_cont):
        embedded = [emb(encoder_cat[..., i]) for i, emb in enumerate(self.embeddings)]
        embedded = torch.cat(embedded, dim=-1)
        combined = torch.cat([embedded, encoder_cont], dim=-1)
        projected = self.input_projection(combined)
        seq_len = projected.size(1)
        projected = projected + self.pos_encoding[:seq_len].unsqueeze(0)
        encoded = self.encoder(projected)
        return self.output_projection(encoded[:, -1, :])


class VanillaTransformerModel(pl.LightningModule):
    def __init__(self, cat_sizes, n_continuous, d_model=64, nhead=4,
                 num_layers=2, dim_feedforward=128, dropout=0.1,
                 prediction_length=24, learning_rate=0.001):
        super().__init__()
        self.save_hyperparameters()
        self.net = VanillaTransformerNet(
            cat_sizes=cat_sizes, n_continuous=n_continuous, d_model=d_model,
            nhead=nhead, num_layers=num_layers, dim_feedforward=dim_feedforward,
            dropout=dropout, prediction_length=prediction_length,
        )
        self.learning_rate = learning_rate

    def forward(self, x):
        return self.net(x["encoder_cat"], x["encoder_cont"])

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = torch.sqrt(nn.functional.mse_loss(y_hat, y[0]) + 1e-8)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = torch.sqrt(nn.functional.mse_loss(y_hat, y[0]) + 1e-8)
        self.log("val_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)


# 從checkpoint載入（save_hyperparameters()已經存了cat_sizes等參數，理論上不用再手動傳）
transformer_model = VanillaTransformerModel.load_from_checkpoint(
    r"C:\Users\peggy\Desktop\能源\transformer_checkpoints\best-transformer-epoch=07-val_loss=0.0123.ckpt"
)
transformer_model.eval()
if torch.cuda.is_available():
    transformer_model = transformer_model.cuda()

building_embed_layer = transformer_model.net.embeddings[0]
print("形狀:", building_embed_layer.weight.shape)

In [ ]:
target_normalizer = training_dataset.target_normalizer

all_preds = []
all_true = []

transformer_model.eval()
with torch.no_grad():
    for x, y in test_dataloader:
        if torch.cuda.is_available():
            x = {k: (v.cuda() if torch.is_tensor(v) else v) for k, v in x.items()}
        y_hat_norm = transformer_model(x)
        y_hat_orig = target_normalizer(dict(prediction=y_hat_norm, target_scale=x["target_scale"]))
        all_preds.append(y_hat_orig.cpu())
        all_true.append(y[0])

y_pred = torch.cat(all_preds, dim=0)
y_true = torch.cat(all_true, dim=0)

print("y_pred有NaN:", torch.isnan(y_pred).any().item())
rmse = torch.sqrt(torch.mean((y_pred - y_true) ** 2)).item()
mae = torch.mean(torch.abs(y_pred - y_true)).item()
print(f"Transformer — test set RMSE: {rmse:.4f}, MAE: {mae:.4f}")

### Transformer 的 test_unseen

In [ ]:
from copy import deepcopy

building_embed_layer = transformer_model.net.embeddings[0]
mean_embedding = building_embed_layer.weight.data.mean(dim=0, keepdim=True)
old_weight = building_embed_layer.weight.data
new_weight = torch.cat([old_weight, mean_embedding], dim=0)

new_embed_layer = torch.nn.Embedding(new_weight.shape[0], new_weight.shape[1])
new_embed_layer.weight.data = new_weight
if torch.cuda.is_available():
    new_embed_layer = new_embed_layer.cuda()
transformer_model.net.embeddings[0] = new_embed_layer

patched_encoder_tr = deepcopy(training_dataset._categorical_encoders["building_id"])
unseen_building_ids = test_unseen_full["building_id"].unique()
n_new = 0
for bid in unseen_building_ids:
    if bid not in patched_encoder_tr.classes_:
        patched_encoder_tr.classes_[bid] = new_weight.shape[0] - 1
        n_new += 1
print(f"新增了{n_new}個building_id對應到index {new_weight.shape[0]-1}（應該是28）")

test_unseen_dataset_tr = TimeSeriesDataSet.from_dataset(
    training_dataset,
    test_unseen_full,
    predict=False,
    stop_randomization=True,
    categorical_encoders={"building_id": patched_encoder_tr},
    min_prediction_length=max_prediction_length,
)
print("test_unseen_dataset樣本數：", len(test_unseen_dataset_tr))

test_unseen_dataloader_tr = test_unseen_dataset_tr.to_dataloader(train=False, batch_size=128, num_workers=0)

all_preds_unseen = []
all_true_unseen = []

transformer_model.eval()
with torch.no_grad():
    for x, y in test_unseen_dataloader_tr:
        if torch.cuda.is_available():
            x = {k: (v.cuda() if torch.is_tensor(v) else v) for k, v in x.items()}
        y_hat_norm = transformer_model(x)
        y_hat_orig = target_normalizer(dict(prediction=y_hat_norm, target_scale=x["target_scale"]))
        all_preds_unseen.append(y_hat_orig.cpu())
        all_true_unseen.append(y[0])

y_pred_unseen = torch.cat(all_preds_unseen, dim=0)
y_true_unseen = torch.cat(all_true_unseen, dim=0)

print("y_pred_unseen有NaN:", torch.isnan(y_pred_unseen).any().item())
rmse_unseen = torch.sqrt(torch.mean((y_pred_unseen - y_true_unseen) ** 2)).item()
mae_unseen = torch.mean(torch.abs(y_pred_unseen - y_true_unseen)).item()
print(f"Transformer — test_unseen set RMSE: {rmse_unseen:.4f}, MAE: {mae_unseen:.4f}")